# Lab 12 — Visualising Data with Matplotlib and Seaborn (Starter)

In this lab you’ll turn **validated, analysis-ready data** into **clear, professional visualisations**.

You will:
- Choose appropriate chart types for specific questions
- Build charts with **Matplotlib**
- Enhance charts with **Seaborn**
- Apply basic design principles (clarity, labels, scale, context)
- Export figures to `output/figures/` and describe insights in markdown

> **Tip:** Run each cell top-to-bottom. The notebook is intentionally guided — follow the prompts and fill in the TODOs.


## 1) Setup

### What you’ll do
- Import libraries
- Confirm your environment is ready
- Load the validated dataset from the `data/` folder


In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    seaborn_available = True
except ImportError:
    seaborn_available = False


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("seaborn:", sns.__version__)

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "validated_dataset_m11.csv"
FIG_DIR = PROJECT_ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

df.head()


In [ ]:
df.info()


In [ ]:
# Basic descriptive summary (numeric + categorical)
display(df.describe(include="number"))
display(df.describe(include="object"))


df.describe(include='all').T.head(12)


In [ ]:
# Helpful prep: parse dates if present
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Create a basic numeric-only view for correlation/heatmaps
numeric_df = df.select_dtypes(include="number")
numeric_df.head()


# Key columns we'll use in this lab (adjust if your dataset differs)
numeric_cols = [c for c in ["price_usd", "units_sold", "revenue_usd"] if c in df.columns]
cat_cols = [c for c in ["category", "state"] if c in df.columns]
date_col = "date" if "date" in df.columns else None

print("Numeric columns:", numeric_cols)
print("Categorical columns:", cat_cols)
print("Date column:", date_col)


In [ ]:
# TODO: choose your numeric column
col = "revenue_usd"  # change if needed

plt.figure(figsize=(8, 4))
plt.hist(df[col].dropna(), bins=10)
plt.title(f"Distribution of {col}")
plt.xlabel(col)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


# Matplotlib example: Total revenue by category (bar chart)

rev_by_cat = (
    df.groupby("category", dropna=False)["revenue_usd"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
plt.bar(rev_by_cat.index.astype(str), rev_by_cat.values)
plt.title("Total Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.savefig(os.path.join(FIG_DIR, "total_revenue_by_category.png"), dpi=150)
plt.show()


In [ ]:
# TODO: choose grouping column + metric
group_col = "category"     # e.g., category, state
metric_col = "revenue_usd" # e.g., revenue_usd, units_sold

summary = (
    df.groupby(group_col, dropna=False)[metric_col]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8, 4))
plt.bar(summary.index.astype(str), summary.values)
plt.title(f"Total {metric_col} by {group_col}")
plt.xlabel(group_col)
plt.ylabel(f"Total {metric_col}")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

summary


# Matplotlib example: Revenue over time (line plot)

df_time = df.dropna(subset=["date"]).copy()

rev_by_day = (
    df_time.groupby(df_time["date"].dt.to_period("D"))["revenue_usd"]
           .sum()
           .sort_index()
)

rev_by_day.index = rev_by_day.index.to_timestamp()

plt.figure(figsize=(10, 5))
plt.plot(rev_by_day.index, rev_by_day.values)
plt.title("Revenue Over Time")
plt.xlabel("Date")
plt.ylabel("Revenue (USD)")
plt.tight_layout()

plt.savefig(os.path.join(FIG_DIR, "revenue_over_time.png"), dpi=150)
plt.show()


In [ ]:
x = "price_usd"
y = "units_sold"

plt.figure(figsize=(6, 4))
plt.scatter(df[x], df[y], alpha=0.8)
plt.title(f"{y} vs {x}")
plt.xlabel(x)
plt.ylabel(y)
plt.tight_layout()
plt.show()


# Matplotlib example: Units sold vs revenue (scatter plot)

plt.figure(figsize=(7, 5))
plt.scatter(df["units_sold"], df["revenue_usd"], alpha=0.6)
plt.title("Units Sold vs Revenue")
plt.xlabel("Units Sold")
plt.ylabel("Revenue (USD)")
plt.tight_layout()

plt.savefig(os.path.join(FIG_DIR, "units_sold_vs_revenue.png"), dpi=150)
plt.show()


In [ ]:
if not seaborn_available:
    print("Seaborn not available — skip Section 4 or install seaborn.")
else:
    sns.set_theme()  # simple default theme

    # 4.1 Boxplot: revenue by category
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="category", y="revenue_usd")
    plt.title("Revenue (USD) by Category")
    plt.xlabel("Category")
    plt.ylabel("Revenue (USD)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
if seaborn_available:
    # 4.2 Regplot: relationship with trend line
    plt.figure(figsize=(6, 4))
    sns.regplot(data=df, x="price_usd", y="units_sold")
    plt.title("Units Sold vs Price (with Trend Line)")
    plt.xlabel("Price (USD)")
    plt.ylabel("Units Sold")
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt

# Seaborn example: Revenue distribution by category (boxplot)

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="category", y="revenue_usd")
plt.title("Revenue Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.savefig(os.path.join(FIG_DIR, "revenue_boxplot_by_category.png"), dpi=150)
plt.show()


## 5) Apply visual design principles (refine at least one chart)

Pick **one** chart you already created and refine it.

Consider:
- Is the title informative and specific?
- Are the axis labels clear (units, currency)?
- Is the scale appropriate?
- Is the chart readable at a glance?

In the cell below, refine your **bar chart** (example). You can refine a different chart if you prefer.


In [ ]:
# Seaborn example: Price distribution (histogram)

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="price_usd", kde=True, bins=20)
plt.title("Price Distribution")
plt.xlabel("Price (USD)")
plt.ylabel("Count")
plt.tight_layout()

plt.savefig(os.path.join(FIG_DIR, "price_distribution.png"), dpi=150)
plt.show()


## 6) Export and save figures

Save at least **2 charts** as PNG files in:

- `../output/figures/` (recommended when running from `notebooks/`)
- or `output/figures/` (if running from the repo root)

✅ Requirements:
- Create the folder if it doesn’t exist
- Use clear, descriptive filenames


In [ ]:
# Refine one chart using design principles (clarity, ordering, readability)

rev_by_cat = (
    df.groupby("category", dropna=False)["revenue_usd"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
plt.bar(rev_by_cat.index.astype(str), rev_by_cat.values)
plt.title("Total Revenue by Category (Ordered)")
plt.xlabel("Category")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(os.path.join(FIG_DIR, "total_revenue_by_category_refined.png"), dpi=150)
plt.show()


## 7) Critique and document your charts (markdown)

For **each** saved figure, add a markdown cell that answers:

- What does this chart show?
- Why is this chart type appropriate?
- What might be misinterpreted (limitations)?
- What decision or question could it support?

Add your responses below.


### Chart critique — Example (replace with your own)

**Figure:** `total_revenue_by_category.png`

- **What it shows:**  
- **Why this chart type:**  
- **Potential misinterpretations / limitations:**  
- **So what? (decision/question it supports):**


## AI Reflection Prompt

Before finalising your submission, use an AI assistant of your choice and ask:

> **“What makes a data visualisation misleading, even when the data itself is correct?”**

Reflect on:
- How design choices influence interpretation  
- Which risks you must actively avoid  
- How clarity supports ethical communication  

Use this reflection to review and improve at least one of your charts.


## Wrap-up checklist

Before you submit, confirm you have:
- [ ] Created **at least 3 visualisations** (Matplotlib and/or Seaborn)
- [ ] Exported **at least 2 PNGs** to `output/figures/`
- [ ] Written markdown critique for each saved figure
- [ ] Included a short AI reflection response (optional, but recommended)

You're done ✅
